# Snowpack -> Hydropower -> Electricity Prices

**Question:** Does winter/spring California snowpack predict summer electricity
price behavior through hydroelectric generation?

**Panel (data/processed/panel.csv):** one row per year --
`snowpack_pct` (April 1 snow water content as % of normal), summer day-ahead
price features (`price_mean`, `price_peak`, `price_vol`, `price_vol_hourly`),
summer hydro generation (`hydro_gwh`, `hydro_gwh_eia`), and controls
(`temp_mean_c`, `heat_days_38c`, `demand_mean_mw`, `gas_mean`).

**Price window:** 2016-2018 + 2023-2025 (6 years; CAISO's public archive has a
2019-2022 gap). Walk-forward uses min_train=3 -> 3 genuinely held-out years.

## Setup

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

## Mediation analysis: snowpack -> hydro output -> price volatility

In [ ]:
from src.models import mediation_analysis, first_stage

panel = pd.read_csv(ROOT / 'data' / 'processed' / 'panel.csv', index_col=0)

# primary mediator: CAISO-reported summer hydro generation (no API key needed)
med = mediation_analysis(panel, target='price_vol', mediator='hydro_gwh')
med

### Reading the paths (Baron-Kenny regression mediation)

In [ ]:
import numpy as np
rows = {
    'total effect (c): snowpack -> volatility':  med['total_effect_c'],
    'a path: snowpack -> hydro':                 med['a_path'],
    'b path: hydro -> volatility (joint)':       med['b_mediator'],
    "direct effect (c'), snowpack | hydro":      med['direct_effect_cp'],
}
for k, v in rows.items():
    print(f'{k:48s} {v:+.3f}')
print(f"\nSobel test of indirect effect: z={med['sobel_z']:.2f}, "
      f"p={med['sobel_p']:.3f} (n={med['n']})")
print()
print('NOTE: at n <= 6 the Sobel test is uninformative (p ~ 1.00); the')
print('proportion-mediated figure is NOT reported as a point estimate here.')

**Causal-chain check:** if the snowpack effect runs *through* hydro,
then adding hydro to the regression should shrink the direct snowpack
coefficient (`c'` close to 0) while hydro keeps a significant `b` coefficient.

**This analysis is EXPLORATORY** -- with n <= 6 overlapping years the Sobel
test is uninformative, so the proportion-mediated figure carries no statistical
weight. It is kept only to illustrate the decomposition, not as evidence.

> Mediation takeaway: fill in after running.